# Mini-Project 02 — Will This Be a Strong Sales Week?
**DS5020 · Bayes' theorem and Naive Bayes on real Walmart sales data**

| Role | Name |
|---|---|
| Data Analyst | Muhammad Saad Malik |
| ML Engineer |  |

**How to run:** 
Cells marked `# TODO` are to be filled in. Every task writes its result into `answers`, and the last cell prints the answer sheet.

## 0. Setup

In [8]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.naive_bayes import CategoricalNB
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score

pd.set_option("display.precision", 4)

# Works whether Jupyter was started in the repo root or in notebooks/
ROOT = Path.cwd() if (Path.cwd() / "data").is_dir() else Path.cwd().parent
DATA_PATH = ROOT / "data" / "Walmart.csv"
FIG_DIR = ROOT / "figures"
FIG_DIR.mkdir(exist_ok=True)
assert DATA_PATH.exists(), f"Walmart.csv not found in {DATA_PATH.parent} (see data/README.md)"

answers = {}   # task -> value; printed as the answer sheet at the end

## 1. Event definitions (R1, R2)

Written **before** any computation and used unchanged in every task.

**Sample space (R2):** one outcome = one store-week; $S$ (the sample space) = all 6,435 store-weeks.
*Notation clash:* the brief uses $S$ both for the sample space and for "strong week". Below, `S` always means **strong week**.

| Event | Definition (one store-week) |
|---|---|
| **S** strong week | `Weekly_Sales` ≥ 1.10 × that store's average weekly sales |
| **H** holiday week | `Holiday_Flag` = 1 |
| **Q** holiday season | the week's month is November or December |
| **C** cold week | `Temperature` < 40 °F |
| **F** expensive fuel | `Fuel_Price` ≥ \$3.50 |
| **J** high unemployment | `Unemployment` ≥ 8 % |
| **G** large store | that store's average weekly sales ≥ \$1,000,000 |

S and G depend on the store average. Parts P and B use the average over all 143 weeks; Part N uses the **training weeks only** (R3).

In [10]:
# --- R1: thresholds live here, once ---
STRONG_FACTOR  = 1.10        # S
SEASON_MONTHS  = [11, 12]    # Q
COLD_F         = 40          # C  (strictly below)
FUEL_HIGH      = 3.50        # F  (at least)
UNEMP_HIGH     = 8.0         # J  (at least)
LARGE_STORE    = 1_000_000   # G  (at least)

EVENTS   = list("SHQCFJG")
FEATURES = ["H", "Q", "C", "F", "J", "G"]    # Naive Bayes inputs; S is the target


def add_events(d, store_avg):
    # Add the seven boolean event columns.
    # store_avg: Series indexed by Store. Pass training-only averages in Part N (R3).
    d = d.copy()
    avg = d["Store"].map(store_avg)
    d["S"] = d["Weekly_Sales"] >= STRONG_FACTOR * avg
    d["H"] = d["Holiday_Flag"] == 1
    d["Q"] = d["Date"].dt.month.isin(SEASON_MONTHS)
    d["C"] = d["Temperature"] < COLD_F
    d["F"] = d["Fuel_Price"] >= FUEL_HIGH
    d["J"] = d["Unemployment"] >= UNEMP_HIGH
    d["G"] = avg >= LARGE_STORE
    return d


# Holiday weeks from Section 2 of the brief (needed for B3)
HOLIDAY_DATES = {
    "Super Bowl":   ["2010-02-12", "2011-02-11", "2012-02-10"],
    "Labor Day":    ["2010-09-10", "2011-09-09", "2012-09-07"],
    "Thanksgiving": ["2010-11-26", "2011-11-25"],
    "Christmas":    ["2010-12-31", "2011-12-30"],
}
DATE_TO_HOLIDAY = {pd.Timestamp(day): name for name, days in HOLIDAY_DATES.items() for day in days}
GROUPS = list(HOLIDAY_DATES) + ["No holiday"]

# print(DATE_TO_HOLIDAY)
print(GROUPS)

['Super Bowl', 'Labor Day', 'Thanksgiving', 'Christmas', 'No holiday']


### Helpers: probability by counting, and the R5 check
`A` and `B` can be a column name (`"S"`) or a boolean Series (`ev.S & ev.H`, `~ev.H`).

In [11]:
def _col(d, A):
    return d[A] if isinstance(A, str) else A

def P(d, A):
    # P(A) = #A / #rows
    return _col(d, A).mean()

def P_given(d, A, B):
    # P(A | B): count inside the rows where B is true
    a, b = _col(d, A), _col(d, B)
    assert b.sum() > 0, "conditioning event is empty"
    return a[b].mean()

def check(x, y, label="", tol=1e-9):
    # R5: two ways of computing the same probability must agree
    assert abs(x - y) < tol, f"{label}: {x} vs {y}"
    print(f"OK  {label}: {x:.4f} = {y:.4f}")

## 2. Warm-up (W1–W7) — on paper first

Do W1–W7 by hand. Afterwards you can rebuild the 20 made-up store-weeks below and check your paper answers with `P` and `P_given`.

In [12]:
# (S, H, Q): number of store-weeks, from the table in Section 4
WARMUP_COUNTS = {
    (1, 1, 1): 2, (1, 1, 0): 1, (1, 0, 1): 2, (1, 0, 0): 1,
    (0, 1, 1): 0, (0, 1, 0): 1, (0, 0, 1): 1, (0, 0, 0): 12,
}
wu = pd.DataFrame([k for k, n in WARMUP_COUNTS.items() for _ in range(n)],
                  columns=["S", "H", "Q"]).astype(bool)
assert len(wu) == 20 and wu.S.sum() == 6 and wu.H.sum() == 4 and wu.Q.sum() == 5

# TODO (optional): check your paper answers, e.g. P(wu, "S"), P_given(wu, "S", "H"), ...

## Part P — The probability of each event
### P1 · Load and check the data

In [13]:
df = pd.read_csv(DATA_PATH)
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True)

n_rows, n_stores, n_weeks = len(df), df["Store"].nunique(), df["Date"].nunique()
n_hol = int(df["Holiday_Flag"].sum())
print(f"{n_rows} rows, {n_stores} stores, {n_weeks} weeks, {n_hol} holiday store-weeks")

assert (n_rows, n_stores, n_weeks, n_hol) == (6435, 45, 143, 450)
assert df.isna().sum().sum() == 0
assert df.groupby("Store").size().eq(143).all()          # every store has every week
assert (df["Date"].dt.day_of_week == 4).all()               # all Fridays -> day/month parsed correctly
assert set(df.loc[df.Holiday_Flag == 1, "Date"]) == set(DATE_TO_HOLIDAY)

answers["P1"] = f"{n_rows} rows, {n_stores} stores, {n_weeks} weeks, {n_hol} holiday store-weeks"
df.head()

6435 rows, 45 stores, 143 weeks, 450 holiday store-weeks


,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
0,1,2010-02-05,1.6437e+06,0,42.31,2.572,211.0964,8.106
1,1,2010-02-12,1.6420e+06,1,38.51,2.548,211.2422,8.106
2,1,2010-02-19,1.6120e+06,0,39.93,2.514,211.2891,8.106
3,1,2010-02-26,1.4097e+06,0,46.63,2.561,211.3196,8.106
4,1,2010-03-05,1.5548e+06,0,46.50,2.625,211.3501,8.106


### P2 · Probability of each event, checked with the complement (R5)
Store averages over **all** weeks here (Parts P and B).

In [16]:
add_events(df, df.groupby("Store")["Weekly_Sales"].mean()).head()

,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment,S,H,Q,C,F,J,G
0,1,2010-02-05,1.6437e+06,0,42.31,2.572,211.0964,8.106,False,False,False,False,False,True,True
1,1,2010-02-12,1.6420e+06,1,38.51,2.548,211.2422,8.106,False,True,False,True,False,True,True
2,1,2010-02-19,1.6120e+06,0,39.93,2.514,211.2891,8.106,False,False,False,True,False,True,True
3,1,2010-02-26,1.4097e+06,0,46.63,2.561,211.3196,8.106,False,False,False,False,False,True,True
4,1,2010-03-05,1.5548e+06,0,46.50,2.625,211.3501,8.106,False,False,False,False,False,True,True


In [14]:
ev = add_events(df, df.groupby("Store")["Weekly_Sales"].mean())

event_table = pd.DataFrame({
    "#E":     [int(ev[e].sum()) for e in EVENTS],
    "#E^c":   [int((~ev[e]).sum()) for e in EVENTS],
    "P(E)":   [P(ev, e) for e in EVENTS],
    "P(E^c)": [P(ev, ~ev[e]) for e in EVENTS],
}, index=EVENTS)
event_table["P(E)+P(E^c)"] = event_table["P(E)"] + event_table["P(E^c)"]

assert (event_table["#E"] + event_table["#E^c"] == len(ev)).all()    # R5 by counting
assert np.allclose(event_table["P(E)+P(E^c)"], 1)                     # R5 by complement
check(P(ev, "H"), 450 / 6435, "P(H) = 450/6435")

answers["P2"] = event_table["P(E)"].round(4).to_dict()
event_table

OK  P(H) = 450/6435: 0.0699 = 0.0699


,#E,#E^c,P(E),P(E^c),P(E)+P(E^c)
S,814,5621,0.1265,0.8735,1.0
H,450,5985,0.0699,0.9301,1.0
Q,810,5625,0.1259,0.8741,1.0
C,989,5446,0.1537,0.8463,1.0
F,3029,3406,0.4707,0.5293,1.0
J,3022,3413,0.4696,0.5304,1.0
G,3146,3289,0.4889,0.5111,1.0


### P3 · Joint probabilities

In [24]:
# TODO P3: joint probabilities by counting, e.g. P(ev, ev.S & ev.H)
joint = {
    "P(S∩H)": P(ev, ev.S & ev.H),
    "P(S∩Q)": P(ev, ev.S & ev.Q),
    "P(H∩Q)": P(ev, ev.H & ev.Q),
    "P(S∩C)": P(ev, ev.S & ev.C)
}
# TODO R5: check each one a second way (e.g. count / 6435, or P(A|B)·P(B))
assert joint["P(S∩H)"] == (ev.S & ev.H).sum() / 6435
assert joint["P(S∩Q)"] == (ev.S & ev.Q).sum() / 6435
assert joint["P(H∩Q)"] == (ev.H & ev.Q).sum() / 6435
assert joint["P(S∩C)"] == (ev.S & ev.C).sum() / 6435

answers["P3"] = joint
joint

{'P(S∩H)': np.float64(0.01895881895881896),
 'P(S∩Q)': np.float64(0.05174825174825175),
 'P(H∩Q)': np.float64(0.027972027972027972),
 'P(S∩C)': np.float64(0.028749028749028748)}

## Part B — Conditional probability and Bayes' theorem
### B1 · Holidays

In [ ]:
# TODO B1: count inside each group
p_S_H  = None    # P(S | H)
p_S_Hc = None    # P(S | H^c)

answers["B1"] = (p_S_H, p_S_Hc)

_B1 sentence:_ …

### B2 · Reverse it with Bayes
$$P(H \mid S) = \frac{P(S \mid H)\,P(H)}{P(S)}$$

In [ ]:
# TODO B2
p_H_S_bayes = None   # from p_S_H, P(H), P(S)
p_H_S_count = None   # count inside the strong weeks
# check(p_H_S_bayes, p_H_S_count, "P(H|S): Bayes vs counting")

answers["B2"] = (p_H_S_bayes, p_H_S_count)

### B3 · Total probability over the five holiday groups
$$P(S) = \sum_g P(S \mid g)\,P(g)$$

In [ ]:
ev["group"] = ev["Date"].map(DATE_TO_HOLIDAY).fillna("No holiday")
assert (ev["group"] != "No holiday").sum() == 450

b3 = pd.DataFrame(index=GROUPS, columns=["n", "P(g)", "P(S|g)", "P(S|g)P(g)", "P(g|S)"], dtype=float)
# TODO B3: fill one row per group g (loop over GROUPS; ev["group"] == g is the event g)

# TODO R5: b3["P(g)"].sum() == 1,  b3["P(S|g)P(g)"].sum() == P(ev, "S"),  b3["P(g|S)"].sum() == 1

answers["B3"] = None   # (holiday with highest P(S|g), group with largest P(g|S))
b3

### B4 · Season

In [ ]:
# TODO B4
p_S_Q = None   # P(S | Q)
p_Q_S = None   # P(Q | S)

answers["B4"] = (p_S_Q, p_Q_S)

_B4 — why are they different?_ …

### B5 · The staffing rules: precision and recall
For a rule "extra staff when event $A$":

- precision $= P(S \mid A) = \dfrac{P(A \mid S)\,P(S)}{P(A \mid S)\,P(S) + P(A \mid S^c)\,P(S^c)}$ (Bayes + total probability), checked by counting
- recall $= P(A \mid S)$ = the fraction of strong weeks that get extra staff

In [ ]:
def precision_by_bayes(p_A_given_S, p_A_given_Sc, p_S):
    # TODO B5: return P(S | A) using the formula above
    return None


rules = pd.DataFrame(index=["holiday (H)", "Nov–Dec (Q)"],
                     columns=["P(A|S)", "P(A|S^c)", "precision (Bayes)", "precision (count)", "recall", "missed"],
                     dtype=float)
# TODO B5: fill both rows; "missed" = 1 - recall. Check Bayes vs counting (R5).

answers["B5"] = None   # holiday: (precision, recall); Nov–Dec: (precision, recall)
rules

### B6 · Base rate
Keep $P(H \mid S)$ and $P(H \mid S^c)$ from B5 fixed and change only $P(S)$.

In [ ]:
# TODO B6: precision_by_bayes(..., p_S = 2 * P(S)) and (..., p_S = P(S) / 2)
prec_doubled = None
prec_halved  = None

answers["B6"] = (prec_doubled, prec_halved)

### B7 · Independence checks
If $A$ and $B$ are independent, $P(A \cap B) = P(A)\,P(B)$. The ratio $P(S\cap B)\,/\,[P(S)P(B)] = P(S\mid B)/P(S)$ shows how far from independence each pair is (1 = independent).

In [ ]:
b7 = pd.DataFrame(index=[f"(S, {b})" for b in FEATURES],
                  columns=["P(S∩B)", "P(S)P(B)", "ratio"], dtype=float)
# TODO B7: fill the table

b7

_B7 sentence — which events carry information about strong weeks?_ …

## Part N — A Naive Bayes classifier for strong weeks
### N1 · Split by time (R3)
Train on 2010–2011, test on 2012. **Store averages come from the training weeks only**, so S and G in the test set use no 2012 information.

In [ ]:
train_raw = df[df["Date"].dt.year <= 2011]
test_raw  = df[df["Date"].dt.year == 2012]
assert len(train_raw) + len(test_raw) == len(df)

store_avg_train = train_raw.groupby("Store")["Weekly_Sales"].mean()   # R3
train = add_events(train_raw, store_avg_train)
test  = add_events(test_raw,  store_avg_train)
print(f"train: {len(train)} store-weeks ({train.Date.nunique()} weeks)   "
      f"test: {len(test)} store-weeks ({test.Date.nunique()} weeks)")

# TODO N1: report P(S) in each set
p_S_train = None
p_S_test  = None

### N2 · Train by counting (training set only)

In [ ]:
# TODO N2: priors
prior_S, prior_Sc = None, None

# TODO N2: likelihoods P(x=1 | S) and P(x=1 | S^c) for each feature
lik = pd.DataFrame(index=FEATURES,
                   columns=["P(x=1|S)", "P(x=1|S^c)", "P(x=1|S) smoothed", "P(x=1|S^c) smoothed"],
                   dtype=float)
# Note for N5: CategoricalNB(alpha=1) uses Laplace smoothing. For a 0/1 feature that is
#   (count of x=1 in the class + 1) / (class size + 2).
# The priors are NOT smoothed. Keep both versions so N3 can match N5 exactly.

lik

### N3 · Predict by hand
$$\text{score}(S) = P(S)\prod_i P(x_i \mid S), \qquad \text{score}(S^c) = P(S^c)\prod_i P(x_i \mid S^c), \qquad P(S \mid x) = \frac{\text{score}(S)}{\text{score}(S) + \text{score}(S^c)}$$
For a feature with $x_i = 0$, use $P(x_i = 0 \mid S) = 1 - P(x_i = 1 \mid S)$.

In [ ]:
profiles = pd.DataFrame(
    [[1, 1, 1, 0, 0, 1],    # (a) holiday, Nov–Dec, cold, cheap fuel, low unemployment, large store
     [0, 0, 0, 1, 1, 0]],   # (b) no holiday, not Nov–Dec, not cold, expensive fuel, high unemployment, small store
    columns=FEATURES, index=["a", "b"])

# TODO N3: the two scores and the normalization for each profile (show your steps)
post_a = None
post_b = None

answers["N3"] = (post_a, post_b)
profiles

### N4 · Check the naive assumption for (H, Q)

In [ ]:
# TODO N4: on the training set, compare P(H∩Q | S) with P(H|S)·P(Q|S)
lhs = None   # P(H∩Q | S)
rhs = None   # P(H | S) * P(Q | S)

_N4 — why are H and Q not independent?_ …

### N5 · scikit-learn

In [ ]:
X_train, y_train = train[FEATURES].astype(int), train["S"].astype(int)
X_test,  y_test  = test[FEATURES].astype(int),  test["S"].astype(int)

nb = CategoricalNB(alpha=1, min_categories=2).fit(X_train, y_train)
sk_post = nb.predict_proba(profiles)[:, 1]      # column 1 = class S
print("scikit-learn P(S | a), P(S | b):", sk_post.round(4))

# TODO N5: check(sk_post[0], post_a, "profile a: hand vs sklearn"), same for b

### N6 · Evaluate on the test set (2012)

In [ ]:
def evaluate(y, yhat):
    tn, fp, fn, tp = confusion_matrix(y, yhat, labels=[0, 1]).ravel()
    return {"TP": tp, "FP": fp, "FN": fn, "TN": tn,
            "accuracy": accuracy_score(y, yhat),
            # zero_division=0: "never strong" predicts no S, so its precision is 0/0. Say so in words.
            "precision P(S|Ŝ)": precision_score(y, yhat, zero_division=0),
            "recall P(Ŝ|S)": recall_score(y, yhat, zero_division=0)}

preds = {
    "Naive Bayes":       None,                                  # TODO N6: nb.predict(...)
    "never strong":      np.zeros(len(test), dtype=int),
    "strong if holiday": test["H"].astype(int).to_numpy(),
}
results = pd.DataFrame({name: evaluate(y_test, yhat) for name, yhat in preds.items() if yhat is not None}).T

answers["N6"] = results[["accuracy", "precision P(S|Ŝ)", "recall P(Ŝ|S)"]].round(4).to_dict("index")
results

_N6 — the test year has no Thanksgiving or Christmas. How does that affect the results?_ …

### N7 · Threshold (stretch)

In [ ]:
p_test = nb.predict_proba(X_test)[:, 1]     # P(S | x) for every test store-week

# TODO N7: for t in [0.1, 0.2, 0.3, 0.5]: predict S when p_test > t, then precision and recall
n7 = pd.DataFrame(index=[0.1, 0.2, 0.3, 0.5], columns=["precision", "recall", "# predicted S"], dtype=float)
n7

## J1 · Note to Maria
Write the five sentences in `submission/note.txt`: how often stores have a strong week; which holiday and season matter most; the honest truth about the holiday rule (precision and what it misses); whether the model beats the rule; one risk. Store numbers only (R4).

## Answer sheet
Copy these into `submission/answer_sheet.md`.

In [ ]:
pd.Series(answers, name="value").to_frame()